In [1]:
# 必要ライブラリのインストール
# Note:: sslはデフォルトで入っているようでインストール不要(pipで入れようとするとエラーになる)
!pip install -U --quiet httpx
!pip install -U --quiet  openai
!pip install -U --quiet langchain-community langchain-openai langgraph langchain
!pip install -U --quiet python-dotenv

In [ ]:
import xml.etree.ElementTree as ET

from elaws_parser.api.hourei_apiv2 import (
    LawXmlParser,
    convert_xml_to_text,
    extract_sections_from_xml,
    get_lawdata_from_law_id,
    get_lawid_from_lawtitle,
    parse_mainprovision_to_text,
    parse_mainprovision_to_text_v2,
    parse_supplprovision_to_text,
    parse_toc_to_text,
    save_xml_string_to_file,
)

# law_id = get_lawid_from_lawtitle("電気事業法")
# if law_id == None:
#     print("LAW DOES NOT EXISTS")

4

In [ ]:
law_name = "土壌汚染対策法"
law_text = extract_text_from_lawid(law_name)

regulation_name = "土壌汚染対策法施行規則"
regulation_text = extract_text_from_lawid(regulation_name)

ID: 414AC0000000053, Num: 平成十四年法律第五十三号, Title: 土壌汚染対策法
ID: 414CO0000000336, Num: 平成十四年政令第三百三十六号, Title: 土壌汚染対策法施行令
ID: 414M60001000023, Num: 平成十四年環境省令第二十三号, Title: 土壌汚染対策法に基づく指定調査機関及び指定支援法人に関する省令
ID: 414M60001000029, Num: 平成十四年環境省令第二十九号, Title: 土壌汚染対策法施行規則
Number of laws: 4
ID: 414M60001000029, Num: 平成十四年環境省令第二十九号, Title: 土壌汚染対策法施行規則
Number of laws: 1
Processing TableStruct in SubItem
Processing TableStruct in Item
Processing TableStruct in Paragraph
Processing TableStruct in Paragraph


In [ ]:
# OpenAI用の変数設定
import os
import ssl

import httpx
import openai
from dotenv import load_dotenv
from langchain_openai import AzureChatOpenAI, ChatOpenAI
from openai import AzureOpenAI, OpenAI

load_dotenv(verbose=True)
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")


def get_llm(modelname: str):

    # https://python.langchain.com/api_reference/openai/chat_models/langchain_openai.chat_models.base.ChatOpenAI.html
    reasoning = {
        "effort": "medium",  # 'low', 'medium', or 'high'
        "summary": "auto",  # 'detailed', 'auto', or None
    }

    # Azureの呼び出しは基本同じ変数も言える
    # https://python.langchain.com/api_reference/openai/chat_models/langchain_openai.chat_models.azure.AzureChatOpenAI.html
    llm = ChatOpenAI(
        model=modelname,
        temperature=1.0,
        max_tokens=None,
        timeout=None,
        max_retries=2,
        api_key=OPENAI_API_KEY,
        # base_url = "https://apim-tepcopgaiservice-dev-01.azure-api.net",
    )
    return llm


llm = get_llm("gpt-5")

# * 法令，施行規則の両方を読み混んで正解を出すパターン

もしもプロンプト長が事足りるならばこれが一番良い．


In [ ]:
# 法令の指定
from langchain.prompts import PromptTemplate, load_prompt

law_name = "土壌汚染対策法"
law_article = "第3条、４条"
enforcement_regulations_name = "土壌汚染対策法施行規則"
prompt_path = "prompts/v001.yaml"
# prompt_path = "prompts/extract_related_laws_v001.yaml"
law_text = extract_text_from_lawid(law_name)
enforcement_regulations = extract_text_from_lawid(enforcement_regulations_name)
prompt = load_prompt(prompt_path)
formatted_prompt = prompt.format(
    law_name=law_name,
    law_article=law_article,
    law_text=law_text,
    enforcement_regulations=enforcement_regulations,
)
messages = [
    ("system", formatted_prompt),
]
ai_msg = llm.invoke(messages)
ai_msg.pretty_print()

Processing TableStruct in SubItem
Processing TableStruct in Item
Processing TableStruct in Paragraph
Processing TableStruct in Paragraph
================================== Ai Message ==================================

### 官庁への申請が必要な場合
1. 旧有害施設跡地で「調査免除の確認」を取りたい工事
    - 電柱・鉄塔の設置・撤去前に、知事へ確認申請が必要（法3条ただし書・様式第三）

2. 調査報告の期限（120日）に間に合わない工事
    - 正当な事情があるときは、報告期限の延長を申請（120日→延長可）

3. 調査で対象物質を事前に確定したい工事
    - 指定調査機関が知事へ申請し、30日以内に物質の通知を受ける

### 注釈
1. 旧有害施設跡地
    - 使用が廃止された「有害物質使用特定施設」のあった土地（法3条1項）

2. 調査免除の確認
    - 「人の健康被害のおそれがない」旨の確認。主に工場敷地等の限定用途で可（施行規則16条）

3. 指定調査機関
    - 知事等が指定した土壌調査の専門機関（法3条1項）

4. 120日
    - 旧有害施設跡地の調査結果の報告期限（起算日から120日：施行規則1条）

5. 30日
    - 対象物質通知の標準回答期限（申請受理から30日以内：施行規則3条3項）


# 法令から最初に関連する条文を抽出する方法

In [ ]:
import logging
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from enum import Enum
from pathlib import Path
from typing import Any, Dict, List, Literal, Optional, TypedDict

import yaml
from langchain_core.language_models import BaseLLM
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langgraph.graph import END, StateGraph

from elaws_parser.extraction.law_extraction_v2 import (
    LegalDocument,
    LegalExtractionConfig,
    create_legal_extraction_system,
)

# ロギング設定
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

config = LegalExtractionConfig(
    model_name="gpt-5", temperature=0.1, prompts_dir="prompts"
)

# システムの初期化
extraction_system = create_legal_extraction_system(config)

# 法令
law_document = LegalDocument(
    name="土壌汚染対策法",
    content=extract_text_from_lawid("土壌汚染対策法"),
    document_type="law",
)

# 施行規則
regulation_document = LegalDocument(
    name="土壌汚染対策法施行規則",
    content=extract_text_from_lawid("土壌汚染対策法施行規則"),
    document_type="regulation",
)

target_articles = ["第3条", "第4条"]

# 処理の実行
try:
    result = extraction_system.process(
        law_document=law_document,
        regulation_document=regulation_document,
        target_articles=target_articles,
    )

    if result["error_message"]:
        print(f"エラーが発生しました: {result['error_message']}")
    else:
        print("=== 最終要点 ===")
        print(result["final_summary"])

        print("\n=== メタデータ ===")
        for key, value in result["metadata"].items():
            print(f"{key}: {value}")

except Exception as e:
    logger.error(f"処理中に予期しないエラーが発生しました: {e}")

INFO:law_extraction:法令要点抽出処理を開始します
INFO:law_extraction:法令からの関連条文抽出を開始


Processing TableStruct in SubItem
Processing TableStruct in Item
Processing TableStruct in Paragraph
Processing TableStruct in Paragraph


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:law_extraction:法令抽出が完了しました
INFO:law_extraction:施行規則からの関連条文抽出を開始
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:law_extraction:施行規則抽出が完了しました
INFO:law_extraction:最終要点生成を開始
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:law_extraction:要点生成が完了しました
INFO:law_extraction:処理が完了しました。ステージ: ProcessingStage.COMPLETED


=== 最終要点 ===
### 官庁への申請が必要な場合
1. 面積3,000㎡以上の掘削・盛土・埋戻し等を行う工事
    - 4条1項の届出（様式6）を事前提出
2. 有害施設の現/旧跡地で面積900㎡以上の工事
    - 4条1項の届出。面積・深さ等を記載
3. 3条「確認」済み土地で大規模・深い工事
    - 面積900㎡以上や深さ50cm超等は届出
4. 旧有害施設跡地で工事に先立つ調査報告が必要
    - 使用廃止・通知から120日以内に報告
5. 3条ただし書の「健康被害なし」確認を受ける場合
    - 調査免除へ。事前に知事へ申請が必要

### 注釈
1. 土地の形質の変更＝地盤を掘る/埋める/盛る等（例：鉄塔基礎、電柱根掘り、地中線の溝掘り）。
2. 届出先・様式（4条1項）＝都道府県知事（政令市は市長）へ様式6、平面・立面・断面図を添付（施行規則23条）。
3. 記載事項（4条1項）＝氏名住所、場所、面積と深さ等。有害施設の現/旧敷地なら施設の種類・場所・扱った有害物質も記載（施行規則24条）。
4. 面積基準（4条1項）＝一般は3,000㎡、有害物質使用特定施設の現/旧敷地は900㎡（施行規則22条）。
5. 3条「確認」済み土地の届出不要（小規模）その1＝面積が900㎡未満なら届出不要（施行規則21条の4一号）。
6. 同その2＝面積900㎡以上でも、土壌を場外搬出しない・飛散/流出を伴わない・深さ50cm未満の全てを満たすなら届出不要（施行規則21条の4二号イ～ハ）。
7. 3条1項の調査・報告＝旧有害施設跡地での工事前に、使用廃止日や知事通知日から120日以内に調査結果を報告。やむを得ない場合は申請で期限延長可（施行規則1条）。
8. 3条ただし書の確認申請（様式3）＝予定利用から「人の健康被害なし」の確認を知事に申請。対象例：外部者立入不可な工場敷地、事業者住居の敷地、小規模工場の住居敷地、鉱山関係の土地（施行規則16条1～3項）。
9. 用語の平易訳
    - 有害物質使用特定施設＝水質汚濁防止法の「有害物質」を扱う施設。
    - 深さ＝工事で地盤をいじる最深部の深さ（施行規則4条4項の定義に準拠）。

=== メタデータ ===
stage: summary_generation
source_document: 